# CTCNet Colab Notebook

This notebook allows you to run [CTCNet](https://github.com/IVIPLab/CTCNet) for Face Image Super-Resolution using Google Colab. It automatically mounts your Google Drive so you can load your test images and pre-trained models, and programmatically load the models and perform inference in Python instead of using the test.py script.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/IVIPLab/CTCNet
%cd CTCNet

In [ ]:
!pip install -r requirement.txt

In [ ]:
import os
import sys
import torch
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from options.test_options import TestOptions
from data import create_dataset
from models import create_model
from utils import utils

# Set up paths to your Google Drive
# Please modify these paths according to where your data is stored in your Google Drive
DRIVE_ROOT = '/content/drive/MyDrive/CTCNet_Data'
TEST_HR_DIR = os.path.join(DRIVE_ROOT, 'test_HR')
PRETRAINED_MODEL_PATH = os.path.join(DRIVE_ROOT, 'best_pth')
SAVE_DIR = os.path.join(DRIVE_ROOT, 'test_results')

# Create save directory if it doesn't exist
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Test HR Dir: {TEST_HR_DIR}")
print(f"Pretrained Model Path: {PRETRAINED_MODEL_PATH}")
print(f"Save Dir: {SAVE_DIR}")

In [ ]:
# Simulate command line arguments to load the models properly using TestOptions
sys.argv = [
    'test.py',
    '--gpus', '1',
    '--model', 'drn',
    '--name', 'SPARNet_S16_V4_Attn2D',
    '--load_size', '128',
    '--dataset_name', 'single',
    '--dataroot', TEST_HR_DIR,
    '--pretrain_model_path', PRETRAINED_MODEL_PATH,
    '--save_as_dir', SAVE_DIR
]

opt = TestOptions().parse()
opt.num_threads = 0
opt.batch_size = 1
opt.serial_batches = True
opt.no_flip = True

dataset = create_dataset(opt)
model = create_model(opt)

if len(opt.pretrain_model_path) > 0:
    model.load_pretrain_model()
else:
    model.setup(opt)

network = model.netG
network.eval()

In [ ]:
# Run inference directly in the notebook
for i, data in enumerate(tqdm(dataset, desc="Processing Images")):
    inp = data['LR']
    with torch.no_grad():
        output_SR = network(inp)
    
    img_path = data['LR_paths']
    output_sr_img = utils.tensor_to_img(output_SR, normal=True)

    save_path = os.path.join(SAVE_DIR, os.path.basename(img_path[0]))
    save_img = Image.fromarray(output_sr_img)
    save_img.save(save_path)

print("Inference complete! Results saved to:", SAVE_DIR)

In [ ]:
import glob

# Visualize the first result
result_images = glob.glob(os.path.join(SAVE_DIR, '*.*'))
if len(result_images) > 0:
    img_path = result_images[0]
    img = Image.open(img_path)
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Super-Resolved Output')
    plt.show()
else:
    print("No result images found in the save directory.")